In [42]:
!pip install tqdm

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

In [ ]:
ns = {'cap': 'urn:oasis:names:tc:emergency:cap:1.2'}

def get_text(root, tag):
    elem = root.find(f'cap:{tag}', ns)
    return elem.text if elem is not None else None

caminho_csv = "alertas_inmet.csv"

inicio = 45789 #35789
fim = 50963
tamanho_chunk = 50

for bloco_inicio in range(inicio, fim, tamanho_chunk):
    bloco_fim = min(bloco_inicio + tamanho_chunk, fim)
    lista_alertas = []

    print(f"Processando IDs {bloco_inicio} a {bloco_fim - 1}")

    for id_alerta in range(bloco_inicio, bloco_fim):
        url = f"https://apiprevmet3.inmet.gov.br/avisos/rss/{id_alerta}"
        try:
            response = requests.get(url, timeout=5)
            response.encoding = 'utf-8'

            if response.status_code != 200:
                print(f"ID {id_alerta} - Erro HTTP {response.status_code}")
                continue

            root = ET.fromstring(response.text)

            dados_alerta = {
                "id": id_alerta,
                "identificador": get_text(root, 'identifier'),
                "emissor": get_text(root, 'sender'),
                "data_envio": get_text(root, 'sent'),
                "evento": root.find('.//cap:event', ns).text if root.find('.//cap:event', ns) is not None else None,
                "severidade": root.find('.//cap:severity', ns).text if root.find('.//cap:severity', ns) is not None else None,
                "descricao": root.find('.//cap:description', ns).text if root.find('.//cap:description', ns) is not None else None,
                "instrucoes": root.find('.//cap:instruction', ns).text if root.find('.//cap:instruction', ns) is not None else None,
                "municipios": None
            }

            municipios = [
                param.find('cap:value', ns).text
                for param in root.findall('.//cap:parameter', ns)
                if param.find('cap:valueName', ns) is not None and param.find('cap:valueName', ns).text == 'Municipios'
            ]
            if municipios:
                dados_alerta['municipios'] = municipios[0]

            lista_alertas.append(dados_alerta)
            print(f"ID {id_alerta} - OK")

        except Exception as e:
            print(f"ID {id_alerta} - erro: {e}")
            continue

    if lista_alertas:
        df_chunk = pd.DataFrame(lista_alertas)
        write_header = not pd.io.common.file_exists(caminho_csv)
        df_chunk.to_csv(caminho_csv, mode='a', header=write_header, index=False, encoding='utf-8')
        print(f"Chunk {bloco_inicio}-{bloco_fim-1} salvo com {len(df_chunk)} registros")

    time.sleep(1)

In [ ]:
caminho_csv = "alertas_inmet.csv"
inicio = 35789
fim = 50963  # fim é exclusivo, então vai até 50962

df = pd.read_csv(caminho_csv)
ids_esperados = set(range(inicio, fim))
ids_salvos = set(df['id'].dropna().astype(int))
ids_faltando = sorted(list(ids_esperados - ids_salvos))

print(f"Total de IDs faltando: {len(ids_faltando)}")
print("Exemplos de IDs faltantes:", ids_faltando[:20]) 

/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


🔍 Total de IDs faltando: 27
Exemplos de IDs faltantes: [36523, 36602, 36603, 37001, 37726, 37759, 37831, 38178, 38815, 39072, 39109, 39160, 39772, 40816, 41166, 42730, 43328, 44755, 47370, 47398]


In [ ]:
ns = {'cap': 'urn:oasis:names:tc:emergency:cap:1.2'}

def get_text(root, tag):
    elem = root.find(f'cap:{tag}', ns)
    return elem.text if elem is not None else None

caminho_csv = "alertas_inmet.csv"
lista_alertas = []

for id_alerta in ids_faltando:
    url = f"https://apiprevmet3.inmet.gov.br/avisos/rss/{id_alerta}"
    try:
        response = requests.get(url, timeout=5)
        response.encoding = 'utf-8'

        if response.status_code != 200:
            print(f"ID {id_alerta} - Erro HTTP {response.status_code}")
            continue

        root = ET.fromstring(response.text)

        dados_alerta = {
            "id": id_alerta,
            "identificador": get_text(root, 'identifier'),
            "emissor": get_text(root, 'sender'),
            "data_envio": get_text(root, 'sent'),
            "evento": root.find('.//cap:event', ns).text if root.find('.//cap:event', ns) is not None else None,
            "severidade": root.find('.//cap:severity', ns).text if root.find('.//cap:severity', ns) is not None else None,
            "descricao": root.find('.//cap:description', ns).text if root.find('.//cap:description', ns) is not None else None,
            "instrucoes": root.find('.//cap:instruction', ns).text if root.find('.//cap:instruction', ns) is not None else None,
            "municipios": None
        }

        municipios = [
            param.find('cap:value', ns).text
            for param in root.findall('.//cap:parameter', ns)
            if param.find('cap:valueName', ns) is not None and param.find('cap:valueName', ns).text == 'Municipios'
        ]
        if municipios:
            dados_alerta['municipios'] = municipios[0]

        lista_alertas.append(dados_alerta)
        print(f"ID {id_alerta} - OK")

    except Exception as e:
        print(f"ID {id_alerta} - erro: {e}")
        continue

    time.sleep(2)

if lista_alertas:
    df_chunk = pd.DataFrame(lista_alertas)
    df_chunk.to_csv(caminho_csv, mode='a', header=False, index=False, encoding='utf-8')
    print(f"{len(df_chunk)} registros reprocessados e salvos com sucesso.")


ID 36523 - Erro HTTP 500
ID 36602 - Erro HTTP 500
ID 36603 - Erro HTTP 500
ID 37001 - Erro HTTP 500
ID 37726 - Erro HTTP 500
ID 37759 - Erro HTTP 500
ID 37831 - Erro HTTP 500
ID 38178 - Erro HTTP 500
ID 38815 - Erro HTTP 500
ID 39072 - Erro HTTP 500
ID 39109 - Erro HTTP 500
ID 39160 - Erro HTTP 500
ID 39772 - Erro HTTP 500
ID 40816 - Erro HTTP 500
ID 41166 - Erro HTTP 500
ID 42730 - Erro HTTP 500
ID 43328 - Erro HTTP 500
ID 44755 - Erro HTTP 500
ID 47370 - Erro HTTP 500
ID 47398 - Erro HTTP 500
ID 47985 - Erro HTTP 500
ID 48226 - Erro HTTP 500
ID 48255 - Erro HTTP 500
ID 48339 - Erro HTTP 500
ID 48373 - Erro HTTP 500
ID 49281 - Erro HTTP 500
ID 50809 - Erro HTTP 500


In [ ]:
caminho_csv = "alertas_inmet.csv"
df = pd.read_csv(caminho_csv)